In [1]:
from pathlib import Path
import json
from glob import glob
import pandas as pd

In [2]:
ROOT_FOLDER = Path("../../corpora/GeneratedCorpus")
SCENARIOS = ["GRAMMAR SWAP", "LEXICAL SWAP", "TRANSLATION"]
TASKS = ["CAPACITAR", "COMPARTILHAR", "EXPLICAR", "EXPLORAR", "FAZER", "RECOMENDAR", "RECRIAR", "RELATAR"]
LANGUAGES = ["ÁRABE", "ESPANHOL", "FRANCÊS", "GALEGO", "INGLÊS", "MANDARIM", "PB NATIVO", "RUSSO", "ÁRABE"]

In [3]:
def extract_texts_from_file(file_path:Path) -> pd.DataFrame:

    texts = []
    curr_text = ""
    curr_task_type = ""

    with open(file_path, "r") as file:
        while (line:= file.readline()) != '':
            
            if line.startswith("<"):
                # All corupus metadata start with '<'. This is used to now when a new corpus begin
                texts.append((curr_task_type, curr_text.strip()))
                
                # Getting the ED, FD, EM or FM
                try:
                    curr_task_type = line.split("_")[1]
                except:
                    print(file_path)

                while (line := file.readline()).startswith("<"): pass
                curr_text = line # first line of the text block


            elif "---" in line:
                # Consume the token '---' and the white lines in between sentences
                while (line:= file.readline()) == " ":pass

            else:
                curr_text += line
        
        # Adding the last text. It is marked by the EOF since there are no other < to indicate new corpus
        texts.append((curr_task_type, curr_text.strip()))

    # Remove any white text that was left            
    texts = list(filter(lambda x: len(x[1])>0, texts))

    return texts


def get_task_and_language(file_path):
    language = None
    for l in LANGUAGES:
        if l in file_path:
            language = l
            break
    
    task = None
    for t in TASKS:
        if t in file_path:
            task = t
            break
    
    return task, language

def extract_text_from_scenario(scenario_path, scenario):
    
    corpus = {
        "scenario": [],
        "language": [],
        "task": [],
        "task_type": [],
        "text": []
    }

    files = glob(scenario_path)

    for file_path in files:
        task, language = get_task_and_language(file_path)

        extracted_texts = extract_texts_from_file(file_path)

        for task_type, text in extracted_texts:

            if task_type not in ("FD", "ED"):
                print(f"{task_type} - {file_path}")

            corpus["scenario"].append(scenario)
            corpus["language"].append(language)
            corpus["task"].append(task)
            corpus["task_type"].append(task_type)
            corpus["text"].append(text)

    return corpus


In [4]:
grammar_swap = extract_text_from_scenario("../../corpora/GeneratedCorpus/GRAMMAR SWAP/*.txt", "GRAMMAR SWAP")
lexical_swap = extract_text_from_scenario("../../corpora/GeneratedCorpus/LEXICAL SWAP/*.txt", "LEXICAL SWAP")
translation = extract_text_from_scenario("../../corpora/GeneratedCorpus/TRANSLATION/*.txt", "TRANSLATION")
pb_nativo = extract_text_from_scenario("../../corpora/GeneratedCorpus/PB NATIVO/*.txt", "PB NATIVO")
pb_nativo_permuted = extract_text_from_scenario("../../corpora/GeneratedCorpus/PB NATIVO - PERMUTED/*.txt", "PB PERMUTED")
pb_nativo_corrupted = extract_text_from_scenario("../../corpora/GeneratedCorpus/PB NATIVO - CORRUPTED/*.txt", "PB CORRUPTED")


In [ ]:
result = {
    key: 
    grammar_swap[key] 
    + lexical_swap[key] 
    + translation[key] 
    + pb_nativo[key] 
    + pb_nativo_permuted[key] 
    + pb_nativo_corrupted[key]
    for key in grammar_swap
}

df = pd.DataFrame(result)

df["task_type"].value_counts()

In [ ]:
df.to_parquet("../../corpora/GeneratedCorpus/corpora.parquet")

In [9]:
def check_imbalanced_scenario(scenario):
    print(f"SCENARIO: {scenario}")

    counts = (
        df[df["scenario"] == scenario]
        .groupby(["task", "language"])
        .size()
        .rename("count")
        .reset_index()
    )

    different = counts.groupby("task").filter(
        lambda group: group["count"].nunique() > 1
    )

    print(different)

In [10]:
check_imbalanced_scenario("TRANSLATION")

SCENARIO: TRANSLATION
Empty DataFrame
Columns: [task, language, count]
Index: []


In [11]:
check_imbalanced_scenario("LEXICAL SWAP")

SCENARIO: LEXICAL SWAP
       task  language  count
42  RECRIAR  ESPANHOL      9
43  RECRIAR   FRANCÊS      9
44  RECRIAR    GALEGO      9
45  RECRIAR    INGLÊS      9
46  RECRIAR  MANDARIM      5
47  RECRIAR     RUSSO      9
48  RECRIAR     ÁRABE      9


In [12]:
check_imbalanced_scenario("GRAMMAR SWAP")

SCENARIO: GRAMMAR SWAP
Empty DataFrame
Columns: [task, language, count]
Index: []
